# TN 2026 Election Analysis Lab

This notebook analyzes the official public ECI result snapshot and tests a hypothetical district-level electoral-college model. The district model is a thought experiment, not the real Indian electoral system.

## 1. Load Data And Assumptions

In [ ]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'

TOTAL_SEATS = 234
MAJORITY_MARK = 118
DATA_VERSION = 'official_public_snapshot_2026_05_05_1618'

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

current_results = read_csv(DATA_DIR / 'current_results_clean.csv')
district_mapping = read_csv(DATA_DIR / 'district_constituency_mapping.csv')
candidate_results = read_csv(DATA_DIR / 'current_candidate_results.csv')

len(current_results), len(district_mapping), len(candidate_results)

## 2. Validate Snapshot

In [ ]:
constituency_numbers = sorted(int(row['constituency_no']) for row in current_results)
assert len(current_results) == TOTAL_SEATS
assert constituency_numbers == list(range(1, TOTAL_SEATS + 1))
assert all(row['source_url'] for row in current_results)
assert all(row['source_last_updated'] for row in current_results)
assert all(row['data_version'] == DATA_VERSION for row in current_results)

chennai_count = sum(1 for row in current_results if row['district'] == 'Chennai')
assert chennai_count == 16

for row in current_results:
    margin = int(row['winner_votes']) - int(row['runnerup_votes'])
    assert margin == int(row['margin']), row['constituency_no']

print('Snapshot validation passed')
print('Source last updated:', current_results[0]['source_last_updated'])
print('Chennai administrative constituencies:', chennai_count)

## 3. Actual Result Summary

In [ ]:
party_counts = Counter(row['winner_party'] for row in current_results)
actual_summary = []
for party, seats in party_counts.most_common():
    actual_summary.append({
        'party': party,
        'seats_won': seats,
        'seat_share': round(seats / TOTAL_SEATS, 4),
        'majority_gap': seats - MAJORITY_MARK,
    })

actual_summary

In [ ]:
tvk_actual = party_counts['TVK']
print(f"TVK actual result: {tvk_actual} / {TOTAL_SEATS}")
print(f"Majority mark: {MAJORITY_MARK}")
print(f"Actual majority gap: {tvk_actual - MAJORITY_MARK}")

## 4. Close Races

In [ ]:
candidate_by_constituency = defaultdict(list)
for row in candidate_results:
    candidate_by_constituency[row['constituency_no']].append(row)

close_races = sorted(current_results, key=lambda row: int(row['margin']))[:15]
for row in close_races:
    ranked = sorted(
        candidate_by_constituency[row['constituency_no']],
        key=lambda candidate: int(candidate['total_votes']),
        reverse=True,
    )
    top_three = [f"{candidate['party']} {int(candidate['total_votes']):,}" for candidate in ranked[:3]]
    others_votes = sum(int(candidate['total_votes']) for candidate in ranked[3:])
    print(
        row['constituency_no'],
        row['constituency_name'],
        'margin',
        f"{int(row['margin']):,}",
        '| top 3:',
        '; '.join(top_three),
        '| others:',
        f"{others_votes:,}",
    )

## 5. District Concentration

In [ ]:
districts = defaultdict(list)
for row in current_results:
    districts[row['district']].append(row)

district_summary = []
for district, rows in sorted(districts.items()):
    counts = Counter(row['winner_party'] for row in rows)
    winner, winner_seats = counts.most_common(1)[0]
    district_summary.append({
        'district': district,
        'constituencies': len(rows),
        'district_leader': winner,
        'leader_seats': winner_seats,
        'tvk_seats': counts.get('TVK', 0),
        'party_counts': dict(counts),
    })

for row in sorted(district_summary, key=lambda item: (-item['tvk_seats'], item['district']))[:12]:
    print(row)

## 6. Hypothetical District Electoral-College Scenario

In [ ]:
def allocate_district_electoral_votes(district_rows, party_col='winner_party', tie_rule='split_by_mla_count'):
    total_votes = len(district_rows)
    party_counts = Counter(row[party_col] for row in district_rows)
    max_count = max(party_counts.values())
    winners = [party for party, count in party_counts.items() if count == max_count]

    if len(winners) == 1:
        return {winners[0]: total_votes}

    if tie_rule == 'split_by_mla_count':
        return dict(party_counts)

    raise ValueError(f'Unsupported tie rule: {tie_rule}')

scenario_totals = Counter()
scenario_by_district = []
for district, rows in sorted(districts.items()):
    allocation = allocate_district_electoral_votes(rows)
    scenario_totals.update(allocation)
    counts = Counter(row['winner_party'] for row in rows)
    scenario_by_district.append({
        'district': district,
        'electoral_votes': len(rows),
        'actual_counts': dict(counts),
        'allocated_votes': allocation,
        'tie': len(allocation) > 1,
    })

scenario_totals

In [ ]:
tvk_hypothetical = scenario_totals['TVK']
print(f"TVK actual seats: {tvk_actual}")
print(f"TVK hypothetical electoral votes: {tvk_hypothetical}")
print(f"Hypothetical majority gap: {tvk_hypothetical - MAJORITY_MARK}")
print('Outcome:', 'majority' if tvk_hypothetical >= MAJORITY_MARK else 'no majority')

In [ ]:
for row in scenario_by_district:
    actual_tvk = row['actual_counts'].get('TVK', 0)
    allocated_tvk = row['allocated_votes'].get('TVK', 0)
    amplification = allocated_tvk - actual_tvk
    if amplification:
        print(row['district'], 'TVK actual', actual_tvk, 'allocated', allocated_tvk, 'amplification', amplification)

## 7. Reconciliation With Form-20

This notebook uses ECI's public result pages as `official_public_snapshot_2026_05_05_1618`. When final Form-20 data is available, rerun the import pipeline and compare constituency candidate totals, margins, and vote shares against this snapshot before using the data as final archival results.